# Week 36, Day 5: Dependency injection

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w036d5.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
class Greeter:
    def __init__(self, clock, out):
        self.clock, self.out = clock, out

    def greet(self, name):
        part = "morning" if self.clock() < 12 else "afternoon"
        self.out(f"Good {part}, {name}")

lines = []
Greeter(clock=lambda: 9, out=lines.append).greet("Ada")
Greeter(clock=lambda: 15, out=lines.append).greet("Bo")
print(lines)
Greeter(clock=lambda: 20, out=print).greet("Chi")

In [ ]:
from datetime import datetime

class Stamp:
    def __init__(self, clock=None):
        self.clock = clock or datetime.now

    def label(self, text):
        return f"[{self.clock():%H:%M}] {text}"

print(Stamp(clock=lambda: datetime(2026, 11, 3, 9, 5)).label("opened"))
print(Stamp().label("opened")[:1] == "[")

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Settings:
    storage: str = "memory"
    data_path: str = "library.json"
    loan_days: int = 14

    @classmethod
    def from_env(cls, env):
        storage = env.get("LIBRARY_STORAGE", "memory")
        if storage not in ("memory", "json"):
            raise ValueError(f"LIBRARY_STORAGE must be memory or json, not {storage!r}")
        days = env.get("LIBRARY_LOAN_DAYS", "14")
        if not days.isdigit() or int(days) < 1:
            raise ValueError("LIBRARY_LOAN_DAYS must be a whole number of days")
        return cls(storage, env.get("LIBRARY_DATA", "library.json"), int(days))

class MemoryBooks:
    def __init__(self):
        self.books = {}

class JsonBooks:
    def __init__(self, path):
        self.path = path

class LoanService:
    def __init__(self, books, loan_days):
        self.books, self.loan_days = books, loan_days

def build(settings):                                  # the composition root
    books = JsonBooks(settings.data_path) if settings.storage == "json" else MemoryBooks()
    return LoanService(books, settings.loan_days)

for env in [{}, {"LIBRARY_STORAGE": "json", "LIBRARY_LOAN_DAYS": "21"}]:
    service = build(Settings.from_env(env))
    print(type(service.books).__name__, service.loan_days)
try:
    Settings.from_env({"LIBRARY_STORAGE": "cloud"})
except ValueError as err:
    print("ValueError:", err)

In [ ]:
class Container:
    def __init__(self):
        self._providers = {}
        self._singletons = {}

    def register(self, name, provider, singleton=False):
        self._providers[name] = (provider, singleton)

    def resolve(self, name):
        provider, singleton = self._providers[name]
        if singleton:
            if name not in self._singletons:
                self._singletons[name] = provider(self)
            return self._singletons[name]
        return provider(self)

class Books:
    def __init__(self):
        self.items = {}

class Service:
    def __init__(self, books, loan_days):
        self.books, self.loan_days = books, loan_days

c = Container()
c.register("loan_days", lambda c: 14)
c.register("books", lambda c: Books(), singleton=True)
c.register("service", lambda c: Service(c.resolve("books"), c.resolve("loan_days")))

a, b = c.resolve("service"), c.resolve("service")
print(a is b, a.books is b.books, a.loan_days)

> **Key idea.** Dependency injection: code receives what it depends on rather than creating or finding it. Prefer constructor injection, which makes the signature honest; use parameters for single calls and `None` defaults for optional dependencies with an obvious real choice. Wire the whole program in one composition root, choosing implementations from a checked, immutable settings object built from the environment (passed in, not read globally). A container registers providers and can share single instances, but belongs only at the edge. Avoid the service locator, which hides dependencies behind a global. Test doubles (fakes, stubs, mocks) stand in for real dependencies.

### Exercise: Making a class injectable

`DailyReport` creates its own dependencies: it reads the real date and prints. Change it to receive `today` (a function returning a `date`, defaulting to `date.today`) and `out` (a function taking a line, defaulting to `print`), keeping the behaviour the same. `run(sales)` writes `Report for 03/11/2026` and then `Total: £12.50` (the sum of the sales in pence).

In [ ]:
from datetime import date

class DailyReport:
    def run(self, sales):
        print(f"Report for {date.today():%d/%m/%Y}")
        total = sum(sales)
        print(f"Total: £{total / 100:.2f}")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_inject():\n    \"injected\"\n    from datetime import date as d\n    import inspect\n    R = ns[\"DailyReport\"]\n    lines = []\n    R(today=lambda: d(2026, 11, 3), out=lines.append).run([1000, 250])\n    assert lines == [\"Report for 03/11/2026\", \"Total: \u00a312.50\"]\n    params = inspect.signature(R).parameters\n    assert params[\"today\"].default is None and params[\"out\"].default is None, \"use None defaults\"\n    import io, contextlib\n    buf = io.StringIO()\n    with contextlib.redirect_stdout(buf):\n        R().run([5])\n    assert buf.getvalue().endswith(\"Total: \u00a30.05\\n\")")

### Exercise: Settings from the environment

Write a frozen dataclass `Settings` with `host` (default `"localhost"`), `port` (default 8080), `debug` (default `False`) and `admins` (default an empty tuple), and a class method `from_env(env)` that reads them from a mapping of strings: `APP_HOST`, `APP_PORT` (a whole number from 1 to 65535), `APP_DEBUG` (`"1"`, `"true"` or `"yes"` in any case mean True; `"0"`, `"false"`, `"no"` or missing mean False; anything else is an error) and `APP_ADMINS` (comma-separated names, spaces stripped, blanks dropped). Any bad value raises `ValueError` naming the variable.

In [ ]:
from dataclasses import dataclass

___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_settings():\n    \"settings\"\n    S = ns[\"Settings\"]\n    s = S.from_env({})\n    assert s == S() and s.port == 8080 and s.debug is False and s.admins == ()\n    s = S.from_env({\"APP_HOST\": \"0.0.0.0\", \"APP_PORT\": \"443\", \"APP_DEBUG\": \"Yes\", \"APP_ADMINS\": \" ada, bo ,,\"})\n    assert (s.host, s.port, s.debug, s.admins) == (\"0.0.0.0\", 443, True, (\"ada\", \"bo\"))\n    for bad, name in [({\"APP_PORT\": \"http\"}, \"APP_PORT\"), ({\"APP_PORT\": \"70000\"}, \"APP_PORT\"), ({\"APP_DEBUG\": \"maybe\"}, \"APP_DEBUG\"), ({\"APP_PORT\": \"0\"}, \"APP_PORT\")]:\n        try:\n            S.from_env(bad)\n            assert False, f\"{bad} should raise ValueError\"\n        except ValueError as err:\n            assert name in str(err), f\"the message should name {name}\"\n    try:\n        s.port = 1\n        assert False, \"settings are frozen\"\n    except AttributeError:\n        pass")

### Exercise: A container that spots loops

Write a `Container` with `register(name, provider, singleton=False)` (registering a name twice raises `ValueError`) and `resolve(name)`. A provider is called with the container. Singletons are made once and shared. An unknown name raises `KeyError`. A **cycle** (a provider that, directly or indirectly, resolves the name being made) must raise `RuntimeError` with a message showing the chain, like `cycle: a -> b -> a`, instead of recursing forever.

In [ ]:
class Container:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_container():\n    \"resolving\"\n    C = ns[\"Container\"]\n    c = C()\n    made = []\n    c.register(\"config\", lambda c: {\"days\": 14})\n    c.register(\"repo\", lambda c: made.append(\"repo\") or object(), singleton=True)\n    c.register(\"service\", lambda c: (c.resolve(\"repo\"), c.resolve(\"config\")[\"days\"]))\n    s1, s2 = c.resolve(\"service\"), c.resolve(\"service\")\n    assert s1[0] is s2[0] and s1[1] == 14 and made == [\"repo\"] and c.resolve(\"config\") is not c.resolve(\"config\")\n    for f, err in [(lambda: c.resolve(\"nope\"), KeyError), (lambda: c.register(\"repo\", lambda c: 1), ValueError)]:\n        try:\n            f()\n            assert False\n        except err:\n            pass\n    c.register(\"a\", lambda c: c.resolve(\"b\"))\n    c.register(\"b\", lambda c: c.resolve(\"a\"))\n    try:\n        c.resolve(\"a\")\n        assert False, \"a cycle should raise RuntimeError\"\n    except RuntimeError as err:\n        assert str(err) == \"cycle: a -> b -> a\", f\"message was {err}\"\n    c.register(\"x\", lambda c: 1)\n    assert c.resolve(\"x\") == 1, \"the container should still work after a cycle error\"")

### Exercise: A composition root

Write `build(settings)`, the composition root for a small shop: it returns an `App` (a simple class you write) with attributes `stock` (a `MemoryStock()` or a `FileStock(path)`, by `settings.storage`), `clock` (`datetime.now`, or a fixed time if `settings.fixed_time` is set) and `checkout`, a `Checkout(stock, clock)`. The classes are given. An unknown storage raises `ValueError`. The same settings must always build the same kinds of object, and two calls to `build` must give independent apps.

In [ ]:
from dataclasses import dataclass
from datetime import datetime

@dataclass(frozen=True)
class Settings:
    storage: str = "memory"
    path: str = "stock.json"
    fixed_time: datetime | None = None

class MemoryStock:
    def __init__(self):
        self.levels = {}

class FileStock:
    def __init__(self, path):
        self.path = path

class Checkout:
    def __init__(self, stock, clock):
        self.stock, self.clock = stock, clock

___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_root():\n    \"wiring\"\n    from datetime import datetime as dt\n    S, M, F, C = ns[\"Settings\"], ns[\"MemoryStock\"], ns[\"FileStock\"], ns[\"Checkout\"]\n    a = build(S())\n    assert isinstance(a.stock, M) and isinstance(a.checkout, C) and a.checkout.stock is a.stock and a.checkout.clock is a.clock\n    assert isinstance(a.clock(), dt)\n    b = build(S(storage=\"file\", path=\"x.json\", fixed_time=dt(2026, 11, 3, 9, 0)))\n    assert isinstance(b.stock, F) and b.stock.path == \"x.json\" and b.clock() == dt(2026, 11, 3, 9, 0) and b.checkout.clock() == dt(2026, 11, 3, 9, 0)\n    assert build(S()).stock is not build(S()).stock\n    try:\n        build(S(storage=\"cloud\"))\n        assert False, \"an unknown storage should raise ValueError\"\n    except ValueError:\n        pass")

### Bug hunt: A default made once

`log` is meant to stamp each message with the **current** hour, but its default is worked out once, when the function is defined, so every message gets the same hour: this prints `09 opened` and `09 closed`. Fix it with a `None` default, so it prints `09 opened` and `17 closed`.

In [ ]:
HOUR = [9]

def current_hour():
    return HOUR[0]

def log(message, hour=current_hour()):
    return f"{hour:02} {message}"

print(log("opened"))
HOUR[0] = 17
print(log("closed"))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"the current hour\"\n    expect_output(\"09 opened\\n17 closed\")\n    assert log(\"x\", 3) == \"03 x\"")

### Bug hunt: A hidden dependency

`Greeting` fetches its clock from a global registry instead of receiving it, so the test's fake clock is ignored and it prints `Good afternoon` (from the registry) instead of `Good morning`. Make `Greeting` take its clock in its constructor, and pass the fake, so the program prints `Good morning`.

In [ ]:
REGISTRY = {"clock": lambda: 15}

class Greeting:
    def __init__(self):
        self.clock = REGISTRY["clock"]

    def text(self):
        return "Good morning" if self.clock() < 12 else "Good afternoon"

fake_clock = lambda: 9
print(Greeting().text())

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"injected\"\n    expect_output(\"Good morning\")\n    assert ns[\"Greeting\"](lambda: 13).text() == \"Good afternoon\"")

### Bug hunt: Two repositories

The repository is registered as an ordinary provider, so the service that adds a book and the report that counts books each get a **different** repository: this prints `0`. It should be one shared instance. Fix the registration so it prints `1`.

In [ ]:
class Container:
    def __init__(self):
        self.providers, self.single = {}, {}
    def register(self, name, provider, singleton=False):
        self.providers[name] = (provider, singleton)
    def resolve(self, name):
        provider, singleton = self.providers[name]
        if not singleton:
            return provider(self)
        if name not in self.single:
            self.single[name] = provider(self)
        return self.single[name]

c = Container()
c.register("books", lambda c: [])
c.register("adder", lambda c: c.resolve("books").append)
c.register("counter", lambda c: lambda: len(c.resolve("books")))

c.resolve("adder")("Aké")
print(c.resolve("counter")())

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"one shared repository\"\n    expect_output(\"1\")")

### Puzzle: An injectable class with a default

Arrange the lines into a class that takes an optional clock, using the real one by default. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
made = []

def make_repo(c):
    made.append("repo")
    return {"n": 0}

providers = {"repo": make_repo}
cache = {}

def resolve(name, shared):
    if shared and name in cache:
        return cache[name]
    obj = providers[name](None)
    if shared:
        cache[name] = obj
    return obj

a = resolve("repo", True)
b = resolve("repo", True)
c = resolve("repo", False)

p = ___   # a is b
q = ___   # a is c
r = ___   # len(made)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert (p, q, r) == (True, False, 2)")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: one program, three set-ups

Wire a small reminders program three ways from settings. The pieces (given) are a `Reminders` service that takes a `store`, a `clock` and a `notifier`, two stores and two notifiers. Write:

- `Settings`, a frozen dataclass with `store` (`"memory"` or `"json"`), `path`, `notify` (`"print"` or `"list"`) and `now` (`None` for the real clock, or text like `"2026-11-03 09:00"` for a fixed one), with `from_env(env)` reading `REM_STORE`, `REM_PATH`, `REM_NOTIFY` and `REM_NOW` (defaults: memory, `reminders.json`, print, the real clock). Bad values raise `ValueError` (a bad `REM_NOW` too).
- `build(settings)`, the composition root, returning the `Reminders` service wired as the settings say. A `"list"` notifier is a `ListNotifier`; the program's tests reach it through `service.notifier.sent`.

The `Reminders` service must not be changed, and must not know which store, clock or notifier it has.

In [ ]:
import json
from dataclasses import dataclass
from datetime import datetime

class MemoryStore:
    def __init__(self):
        self.items = []
    def add(self, when, text):
        self.items.append((when, text))
    def due(self, now):
        due = [i for i in self.items if i[0] <= now]
        self.items = [i for i in self.items if i[0] > now]
        return due

class JsonStore:
    def __init__(self, path):
        self.path = path
    def _load(self):
        try:
            with open(self.path) as f:
                return [(datetime.fromisoformat(w), t) for w, t in json.load(f)]
        except FileNotFoundError:
            return []
    def _save(self, items):
        with open(self.path, "w") as f:
            json.dump([(w.isoformat(), t) for w, t in items], f)
    def add(self, when, text):
        self._save(self._load() + [(when, text)])
    def due(self, now):
        items = self._load()
        self._save([i for i in items if i[0] > now])
        return [i for i in items if i[0] <= now]

class PrintNotifier:
    def send(self, text):
        print("REMINDER:", text)

class ListNotifier:
    def __init__(self):
        self.sent = []
    def send(self, text):
        self.sent.append(text)

class Reminders:
    def __init__(self, store, clock, notifier):
        self.store, self.clock, self.notifier = store, clock, notifier
    def add(self, when, text):
        self.store.add(when, text)
    def run(self):
        due = self.store.due(self.clock())
        for _, text in sorted(due):
            self.notifier.send(text)
        return len(due)

___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_settings():\n    \"settings\"\n    S = ns[\"Settings\"]\n    s = S.from_env({})\n    assert (s.store, s.path, s.notify, s.now) == (\"memory\", \"reminders.json\", \"print\", None)\n    s = S.from_env({\"REM_STORE\": \"json\", \"REM_PATH\": \"r.json\", \"REM_NOTIFY\": \"list\", \"REM_NOW\": \"2026-11-03 09:00\"})\n    assert (s.store, s.path, s.notify, s.now) == (\"json\", \"r.json\", \"list\", \"2026-11-03 09:00\")\n    for bad in [{\"REM_STORE\": \"cloud\"}, {\"REM_NOTIFY\": \"email\"}, {\"REM_NOW\": \"tomorrow\"}]:\n        try:\n            S.from_env(bad)\n            assert False, f\"{bad} should raise ValueError\"\n        except ValueError:\n            pass\ndef test_build():\n    \"three set-ups\"\n    from datetime import datetime as dt\n    S = ns[\"Settings\"]\n    svc = build(S.from_env({\"REM_NOTIFY\": \"list\", \"REM_NOW\": \"2026-11-03 09:00\"}))\n    svc.add(dt(2026, 11, 3, 8, 0), \"medicine\")\n    svc.add(dt(2026, 11, 3, 18, 0), \"call Kemi\")\n    assert svc.run() == 1 and svc.notifier.sent == [\"medicine\"] and type(svc.store).__name__ == \"MemoryStore\"\n    js = build(S.from_env({\"REM_STORE\": \"json\", \"REM_PATH\": \"rem.json\", \"REM_NOTIFY\": \"list\", \"REM_NOW\": \"2026-11-03 19:00\"}))\n    js.add(dt(2026, 11, 3, 18, 0), \"call Kemi\")\n    again = build(S.from_env({\"REM_STORE\": \"json\", \"REM_PATH\": \"rem.json\", \"REM_NOTIFY\": \"list\", \"REM_NOW\": \"2026-11-03 19:00\"}))\n    assert again.run() == 1 and again.notifier.sent == [\"call Kemi\"] and again.run() == 0\n    real = build(S.from_env({}))\n    assert type(real.notifier).__name__ == \"PrintNotifier\" and isinstance(real.clock(), dt)")

## Warm-up

The same class greeted three people at three "times", into a list and onto the screen, without changing a line of it. Everything it depends on (what time it is, where words go) was **injected**. Today scales that idea up from one class to a whole program.

## From one class to a program

**Dependency injection** means a piece of code receives the things it depends on from outside, rather than creating or finding them itself. Week 34 showed why: the code becomes testable with fakes, and the details (the real clock, the real email server) can change without touching it.

In a whole program, objects depend on objects that depend on objects: the command-line interface needs a loan service, which needs a book repository and a member repository and a clock, and the repositories need a file path. Someone has to create all of these, in the right order, and decide **which** implementation of each to use (a JSON repository or a memory one? the real clock or a fixed one?). Today is about doing that well.

## Three ways to inject

| Way | Looks like | Use it for |
|---|---|---|
| **constructor** injection | `LoanService(books, members, clock)` | dependencies an object needs for its whole life: the default choice |
| **parameter** injection | `report(loans, out=print)` | functions, or something needed for one call only |
| **attribute** injection | `service.clock = fake` after making it | rarely; easy to forget, and the object is half-made until it's done |

Constructor injection is the default because it's **honest**: the signature lists everything the class needs, and the object is complete as soon as it exists.

A **default** can make injection convenient without hiding it. Use `None` as the default and choose the real thing inside, so the default is made fresh each time (Week 28's shared-default trap):

Defaults suit **optional** dependencies with an obvious real choice (the system clock). Dependencies with no sensible default, such as which repository to use, should be required arguments.

## The composition root and configuration

All the creating and connecting happens in **one** place, the **composition root** (Week 34): usually a function at the very edge of the program, called by `main`. It's the only code that names concrete classes for the details. **Configuration** tells it which ones to choose.

Configuration usually comes from **environment variables** (settings given to a program by the system it runs on, read in Python from `os.environ`, a dictionary) or a settings file. Turn it into a checked, immutable object once, at the start, rather than reading `os.environ` all over the program:

`from_env` takes the environment **as an argument** (in the real program, `Settings.from_env(os.environ)`), so tests can pass a plain dictionary. That's injection again, applied to configuration. A bad setting fails at start-up with a clear message, not halfway through the night.

Tests rarely use the composition root at all: they build just the part they're testing, with fakes. The composition root itself gets one or two tests checking that it wires the right classes.

## A small container

In a large program the composition root grows long. A **dependency injection container** can take over some of the work: you register, for each name, a **provider** (a function that makes the object), and the container calls providers on demand, passing itself so providers can ask for what they need. It can also keep **one shared instance** of things that should exist once (a repository, a database connection):

Each `service` is new, but both share the single `books` repository. Containers like this exist as libraries for Python, but many Python programs never need one: a plain composition-root function is clearer until the wiring becomes genuinely large. Use a container, if at all, only in the composition root.

## Keeping dependencies visible

The tempting shortcut is to put the container (or a dictionary of shared objects) in a global, and have classes fetch what they need from it:

```python
class LoanService:
    def __init__(self):
        self.books = CONTAINER.resolve("books")       # hidden dependency
```

This is called the **service locator** pattern, and Mark Seemann, among others, describes it as an anti-pattern: the class's signature no longer says what it needs, tests have to set up the global before making one, and nothing stops two tests interfering through it. It's Week 34's hidden global coupling in a new form. Keep dependencies **visible** in signatures, and keep the container (if you have one) at the edge.

A related term you'll meet: **test doubles**, the general name for stand-ins used in tests. A **fake** is a working, simpler version (a memory repository); a **stub** returns fixed answers (a clock that always says 9 o'clock); a **mock** also records how it was called so the test can check (Python's `unittest.mock` provides these, as Week 24 mentioned). Hand-written fakes and stubs, passed in through constructors, cover most needs.

## Practice

## Quiz

## Daily challenge

## Before you go

Look at the `main` function of a program you've written (the Week 25 project is a good one) and pick out its composition root: the lines that create the real objects. Could a test build the same program with fakes? Tomorrow is the Forge: patterns and architecture together in one program.